In [16]:
# !pip install --no-cache-dir pandas pyarrow
# !pip install openpyxl
!pip install tqdm



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [1]:
print("hello world")

hello world


In [2]:
import pandas as pd
import requests
import json
import time
from tqdm import tqdm

In [3]:
gold_company = pd.read_excel("/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/Master_Matching_20260701_1059 (1).xlsx", engine="openpyxl")
gold_company

,Company ID IDX,Company ID BUMN,Nama Perusahaan IDX,Nama Perusahaan BUMN,Nama Perusahaan AHU,Jaro IDX_BUMN,Jaro IDX_AHU,Jaro BUMN_AHU,Sektor,Sub Sektor,...,Kode Emiten (Ticker),Tanggal IPO (Listed),Papan Pencatatan,Alamat Lengkap,Telepon,Fax,Email,Website,NPWP,Pair Category
0,1.001,NaN,PT Adaro Andalan Indonesia Tbk,NaN,PT Adaro Andalan Indonesia,0.0,1.0,0.0,Energi,"Minyak, Gas & Batu Bara",...,AADI,2024-12-05,Utama,Cyber 2 Tower Lantai 26\nJl. H.R. Rasuna Said ...,(021) 2553 3065 | 02125533000,(021) 2553 3066,corsec@adaroindonesia.com,www.adaroindonesia.com,02.433.115.9-091.000,IDX + AHU
1,1.002,NaN,Adaro Australia Pty Ltd,NaN,NaN,0.0,0.0,0.0,Investasi,Investasi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
2,1.003,NaN,Adaro Capital Limited,NaN,NaN,0.0,0.0,0.0,Investasi,Investasi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
3,1.004,NaN,Adaro International (Singapore) Pte Ltd,NaN,NaN,0.0,0.0,0.0,Perdagangan batubara,Perdagangan batubara,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
4,1.005,NaN,Arindo Holdings (Mauritius) Ltd,NaN,NaN,0.0,0.0,0.0,Investasi,Investasi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
772560,NaN,NaN,NaN,NaN,PT Fatima Trading Companies,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,"Gedung Fancy Mampang, Jl. Mampang Prpt. Raya N...",0881024990317,NaN,NaN,NaN,NaN,AHU Only
772561,NaN,NaN,NaN,NaN,PT Simpul Eventworks Indonesia,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,Jl. Wolter Monginsidi No.107A,087784451067,NaN,NaN,NaN,NaN,AHU Only
772562,NaN,NaN,NaN,NaN,PT Sinar Fajar Nusantara Kilat,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,Jalan Ancol Barat VIII Nomor 1 Kontener Nomor ...,081387119253,NaN,NaN,NaN,NaN,AHU Only
772563,NaN,NaN,NaN,NaN,PT Stasiun Hangat Sejahtera,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,"Infiniti Office, Bellezza BSA, 1st Floor Unit ...",082364055502,NaN,NaN,NaN,NaN,AHU Only


In [4]:
gold_company_address = gold_company[["Alamat Lengkap"]]
gold_company_address

,Alamat Lengkap
0,Cyber 2 Tower Lantai 26\nJl. H.R. Rasuna Said ...
1,NaN
2,NaN
3,NaN
4,NaN
...,...
772560,"Gedung Fancy Mampang, Jl. Mampang Prpt. Raya N..."
772561,Jl. Wolter Monginsidi No.107A
772562,Jalan Ancol Barat VIII Nomor 1 Kontener Nomor ...
772563,"Infiniti Office, Bellezza BSA, 1st Floor Unit ..."


In [5]:
#count NaN address
gold_company_address.isnull().sum()

Alamat Lengkap    6862
dtype: int64

In [6]:
df_sample = gold_company_address[
    gold_company_address["Alamat Lengkap"].str.len() > 30
].sample(n=1000, random_state=42)

In [10]:
df_sample

,Alamat Lengkap
771473,"JALAN RAYA CILINCING NO 36, RUKO CILINCING PLA..."
705059,"JALAN CENDRAWASIH PERUMAHAN TAHAP 3 NOMOR 4, K..."
751424,"SEDAYU SQUARE Blok B 12, JALAN OUTER RINGROAD ..."
684392,"GRAHA MULIA, CLUSTER DIAMOND, BLOK P7"
286020,Ruko Thematik Paramount Marketplace Blok P/51 ...
...,...
108374,"4W OFFICE, JALAN RING ROAD BUBULAK NOMOR A-4"
767431,Northwest Boulevard NV 07/6 (Lantai 2)
317657,SPAZIO TOWER LANTAI 11 UNIT 1107 Jl. Mayjend Y...
343277,"WIJAYA GRAND CENTER, JL. WIJAYA II NO.15"


In [ ]:
def parse_address(alamat):
    prompt = f"""
Ekstrak alamat berikut ke JSON.

Aturan:
- Jangan mengarang.
- Jika tidak ada, isi "".
- Semua teks yang tidak masuk field lain masukkan ke remaining_text.
- Output JSON valid saja.

Alamat:
{alamat}

{{
  "gedung": "",
  "lantai": "",
  "jalan": "",
  "no": "nomor ..",
  "kelurahan": "",
  "kecamatan": "",
  "kota": "",
  "provinsi": "",
  "kodepos": "",
  "remaining_text": ""
}}
"""
    
    try:
        r = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "qwen2.5:7b",
                "prompt": prompt,
                "format": "json",
                "stream": False,
                "options": {
                    "temperature": 0,
                    "num_predict": 120,
                    "num_ctx": 512
                }
            },
            timeout=60
        )

        raw = r.json()
        return json.loads(raw["response"])

    except Exception as e:
        print("ERROR:", e)
        return None

In [21]:
import json

sample = df_sample["Alamat Lengkap"].iloc[0]

result = parse_address(sample)

print("=== ORIGINAL ===")
print(sample)

print("\n=== PARSED ===")
print(json.dumps(result, indent=2, ensure_ascii=False))

=== ORIGINAL ===
JALAN RAYA CILINCING NO 36, RUKO CILINCING PLAZZA BLOK A/2

=== PARSED ===
{
  "gedung": "",
  "lantai": "",
  "jalan": "JALAN RAYA CILINCING",
  "no": "36",
  "kelurahan": "",
  "kecamatan": "",
  "kota": "",
  "provinsi": "",
  "kodepos": "",
  "remaining_text": "RUKO CILINCING PLAZZA BLOK A/2"
}


In [22]:
from tqdm import tqdm
import pandas as pd
import time
import json

dataset = []
failed = []

for i, addr in enumerate(tqdm(df_sample["Alamat Lengkap"].head(1000), total=1000)):

    result = parse_address(addr)

    if result is not None:
        dataset.append({
            "text": addr,
            "label": result
        })
    else:
        failed.append({
            "text": addr,
            "idx": i
        })

    # checkpoint tiap 25 data (WAJIB biar nggak nyesel 😭)
    if (i + 1) % 25 == 0:
        pd.DataFrame(dataset).to_json(
            "dataset_checkpoint.json",
            orient="records",
            force_ascii=False,
            indent=2
        )

        pd.DataFrame(failed).to_json(
            "failed_checkpoint.json",
            orient="records",
            force_ascii=False,
            indent=2
        )

    time.sleep(0.02)  # ringan aja, biar Ollama nggak overheat

 32%|███▏      | 322/1000 [37:57<1:21:33,  7.22s/it] 

ERROR: Unterminated string starting at: line 11 column 21 (char 290)


 58%|█████▊    | 583/1000 [1:09:39<56:01,  8.06s/it] 

ERROR: Expecting ',' delimiter: line 12 column 1 (char 293)


 89%|████████▉ | 893/1000 [1:50:02<12:53,  7.23s/it]   

ERROR: Unterminated string starting at: line 11 column 21 (char 218)


100%|██████████| 1000/1000 [2:02:15<00:00,  7.34s/it]


In [23]:
dataset

[{'text': 'JALAN RAYA CILINCING NO 36, RUKO CILINCING PLAZZA BLOK A/2',
  'label': {'gedung': '',
   'lantai': '',
   'jalan': 'JALAN RAYA CILINCING',
   'no': '36',
   'kelurahan': '',
   'kecamatan': '',
   'kota': '',
   'provinsi': '',
   'kodepos': '',
   'remaining_text': 'RUKO CILINCING PLAZZA BLOK A/2'}},
 {'text': 'JALAN CENDRAWASIH PERUMAHAN TAHAP 3 NOMOR 4, KOTA DURI, KABUPATEN BENGKALIS',
  'label': {'gedung': '',
   'lantai': '',
   'jalan': 'JALAN CENDRAWASIH',
   'no': '4',
   'kelurahan': '',
   'kecamatan': '',
   'kota': 'KOTA DURI',
   'provinsi': '',
   'kodepos': '',
   'remaining_text': 'PERUMAHAN TAHAP 3'}},
 {'text': 'SEDAYU SQUARE Blok B 12, JALAN OUTER RINGROAD LKR. LUAR',
  'label': {'gedung': 'SEDAYU SQUARE',
   'lantai': '',
   'jalan': 'JALAN OUTER RINGROAD LKR. LUAR',
   'no': 'Blok B 12',
   'kelurahan': '',
   'kecamatan': '',
   'kota': '',
   'provinsi': '',
   'kodepos': '',
   'remaining_text': ''}},
 {'text': 'GRAHA MULIA, CLUSTER DIAMOND, BLOK P7'

In [24]:
import pandas as pd

df_train = pd.DataFrame(dataset)

df_train.to_parquet(
    "train_address_1k.parquet",
    engine="pyarrow",
    index=False
)

In [25]:
df_train.to_json(
    "train_address_1k.json",
    orient="records",
    force_ascii=False,
    indent=2
)

In [26]:
print(df_train.shape)
print(df_train.head(3))

(997, 2)
                                                text  \
0  JALAN RAYA CILINCING NO 36, RUKO CILINCING PLA...   
1  JALAN CENDRAWASIH PERUMAHAN TAHAP 3 NOMOR 4, K...   
2  SEDAYU SQUARE Blok B 12, JALAN OUTER RINGROAD ...   

                                               label  
0  {'gedung': '', 'lantai': '', 'jalan': 'JALAN R...  
1  {'gedung': '', 'lantai': '', 'jalan': 'JALAN C...  
2  {'gedung': 'SEDAYU SQUARE', 'lantai': '', 'jal...  


In [27]:
df_train

,text,label
0,"JALAN RAYA CILINCING NO 36, RUKO CILINCING PLA...","{'gedung': '', 'lantai': '', 'jalan': 'JALAN R..."
1,"JALAN CENDRAWASIH PERUMAHAN TAHAP 3 NOMOR 4, K...","{'gedung': '', 'lantai': '', 'jalan': 'JALAN C..."
2,"SEDAYU SQUARE Blok B 12, JALAN OUTER RINGROAD ...","{'gedung': 'SEDAYU SQUARE', 'lantai': '', 'jal..."
3,"GRAHA MULIA, CLUSTER DIAMOND, BLOK P7","{'gedung': 'GRAHA MULIA', 'lantai': '', 'jalan..."
4,Ruko Thematik Paramount Marketplace Blok P/51 ...,{'gedung': 'Ruko Thematik Paramount Marketplac...
...,...,...
992,"4W OFFICE, JALAN RING ROAD BUBULAK NOMOR A-4","{'gedung': '4W Office', 'lantai': '', 'jalan':..."
993,Northwest Boulevard NV 07/6 (Lantai 2),"{'gedung': '', 'lantai': '2', 'jalan': 'Northw..."
994,SPAZIO TOWER LANTAI 11 UNIT 1107 Jl. Mayjend Y...,"{'gedung': 'SPAZIO TOWER', 'lantai': '11', 'ja..."
995,"WIJAYA GRAND CENTER, JL. WIJAYA II NO.15","{'gedung': 'WIJAYA GRAND CENTER', 'lantai': ''..."


##DUMP

In [12]:
addr = df_sample["Alamat Lengkap"].iloc[0]

print(addr)

result = parse_address(addr)

print(result)
print(type(result))

JALAN RAYA CILINCING NO 36, RUKO CILINCING PLAZZA BLOK A/2
None
<class 'NoneType'>
